# Lab 14: Singular Value Decomposition

Independent-study notebook with solutions.

In this lab you will compute singular values, reconstruct matrices, form low-rank approximations, use the pseudoinverse for least squares, and connect SVD to PCA.

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

## 2. Compute SVD and reconstruct the matrix

For a matrix $A$, NumPy returns

$$
A = U\Sigma V^T.
$$

In code, `np.linalg.svd(A)` returns `U`, the singular values `s`, and `Vt`.

In [ ]:
A = np.array([[0, 1, 1],
              [1, 1, 0]], dtype=float)

U, s, Vt = np.linalg.svd(A, full_matrices=True)
print("U =
", U)
print("singular values =", s)
print("Vt =
", Vt)

Sigma = np.zeros_like(A, dtype=float)
Sigma[:len(s), :len(s)] = np.diag(s)
A_rec = U @ Sigma @ Vt
print("Reconstructed A =
", A_rec)
print("reconstruction error =", np.linalg.norm(A - A_rec))

### Question 1
What are the singular values of $A$? What is the rank of $A$?

<details>
<summary>Solution</summary>

The singular values are printed above. The rank is the number of nonzero singular values. For this matrix, both singular values are nonzero, so the rank is $2$.

</details>

## 3. Singular values from $A^TA$

Singular values are square roots of eigenvalues of $A^TA$.

In [ ]:
G = A.T @ A
print("A^T A =
", G)
lam, V = np.linalg.eigh(G)
# sort in descending order
idx = np.argsort(lam)[::-1]
lam = lam[idx]
V = V[:, idx]
print("eigenvalues of A^T A =", lam)
print("sqrt eigenvalues =", np.sqrt(np.maximum(lam, 0)))

### Similar practice question
For

$$
B=\begin{bmatrix}1&4\\2&2\\2&-4\end{bmatrix},
$$

compute $B^TB$ and find the singular values.

In [ ]:
B = np.array([[1, 4],
              [2, 2],
              [2,-4]], dtype=float)

G = B.T @ B
print("B^T B =
", G)
print("singular values =", np.linalg.svd(B, compute_uv=False))

<details>
<summary>Solution</summary>

Here

$$
B^TB=\begin{bmatrix}9&0\\0&36\end{bmatrix}.
$$

The eigenvalues are $36$ and $9$, so the singular values are $6$ and $3$.

</details>

## 4. Geometry of SVD

A $2\times2$ matrix maps the unit circle to an ellipse. The semi-axis lengths are the singular values.

In [ ]:
C = np.array([[2.0, 1.0],
              [0.5, 1.5]])

t = np.linspace(0, 2*np.pi, 400)
circle = np.vstack([np.cos(t), np.sin(t)])
ellipse = C @ circle

plt.figure(figsize=(5,5))
plt.plot(circle[0], circle[1], label="unit circle")
plt.plot(ellipse[0], ellipse[1], label="C applied to circle")
plt.axis("equal")
plt.grid(True)
plt.legend()
plt.title("Unit circle under a linear map")
plt.show()

print("singular values of C =", np.linalg.svd(C, compute_uv=False))

### Question 2
What do the singular values tell you about the ellipse?

<details>
<summary>Solution</summary>

The singular values are the lengths of the semi-major and semi-minor axes of the ellipse obtained by applying the matrix to the unit circle.

</details>

## 5. Rank-$k$ approximation

The rank-$k$ truncation is

$$
A_k=\sum_{i=1}^k \sigma_i u_i v_i^T.
$$

In [ ]:
# Synthetic image-like matrix
x = np.linspace(-3, 3, 80)
y = np.linspace(-3, 3, 80)
Xg, Yg = np.meshgrid(x, y)
Image = np.exp(-(Xg**2 + Yg**2)) + 0.5*np.exp(-((Xg-1.2)**2 + (Yg+0.7)**2)/0.4)

U, s, Vt = np.linalg.svd(Image, full_matrices=False)

def rank_k_approx(k):
    return U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]

for k in [1, 3, 5, 10, 20]:
    Ak = rank_k_approx(k)
    rel_error = np.linalg.norm(Image - Ak, 'fro') / np.linalg.norm(Image, 'fro')
    print(f"k={k:2d}, relative Frobenius error={rel_error:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, k in zip(axes, [1, 3, 10, 20]):
    ax.imshow(rank_k_approx(k), cmap='gray')
    ax.set_title(f"rank {k}")
    ax.axis('off')
plt.show()

### Question 3
Why does the rank-$k$ approximation improve as $k$ increases?

<details>
<summary>Solution</summary>

Increasing $k$ keeps more singular directions. Since the SVD orders singular values from largest to smallest, each added term restores the next largest remaining component of the matrix.

</details>

## 6. Pseudoinverse and least squares

The SVD gives the Moore--Penrose pseudoinverse $A^+$.

In [ ]:
A = np.array([[1, 1],
              [1, 2],
              [1, 3]], dtype=float)
b = np.array([1, 2, 2], dtype=float)

x_pinv = np.linalg.pinv(A) @ b
x_lstsq, residuals, rank, s = np.linalg.lstsq(A, b, rcond=None)

print("pseudoinverse least-squares solution =", x_pinv)
print("np.linalg.lstsq solution =", x_lstsq)
print("residual norm =", np.linalg.norm(A @ x_pinv - b))

### Similar practice question
Use the pseudoinverse to solve the least-squares problem for

$$
A=\begin{bmatrix}1&0\\1&1\\1&2\\1&3\end{bmatrix},
\qquad
b=\begin{bmatrix}1\\2\\2\\4\end{bmatrix}.
$$

In [ ]:
A2 = np.array([[1,0],[1,1],[1,2],[1,3]], dtype=float)
b2 = np.array([1,2,2,4], dtype=float)
x2 = np.linalg.pinv(A2) @ b2
print("solution =", x2)
print("residual norm =", np.linalg.norm(A2 @ x2 - b2))

<details>
<summary>Solution interpretation</summary>

The first coordinate is the intercept and the second coordinate is the slope of the best-fit line $y=\beta_0+\beta_1x$.

</details>

## 7. PCA using SVD

For a centered data matrix $X$, the right singular vectors are the principal component directions.

In [ ]:
np.random.seed(1)
Z = np.random.randn(300, 2)
X = Z @ np.array([[3, 1], [0, 0.6]])
X = X - X.mean(axis=0)

U, s, Vt = np.linalg.svd(X, full_matrices=False)
print("principal directions as rows of Vt =
", Vt)
print("explained variance proportions =", s**2 / np.sum(s**2))

plt.figure(figsize=(5,5))
plt.scatter(X[:,0], X[:,1], alpha=0.35)
origin = np.array([0,0])
for i in range(2):
    direction = Vt[i]
    length = s[i] / np.sqrt(len(X))
    plt.arrow(0, 0, length*direction[0], length*direction[1], width=0.03, length_includes_head=True)
plt.axis('equal')
plt.grid(True)
plt.title("PCA directions from SVD")
plt.show()

### Question 4
Why does PCA use the right singular vectors of the centered data matrix?

<details>
<summary>Solution</summary>

For centered data $X$, the covariance matrix is proportional to $X^TX$. The principal directions are eigenvectors of the covariance matrix, hence eigenvectors of $X^TX$. These are exactly the right singular vectors of $X$.

</details>

## 8. Final reflection

Write short answers to the following.

1. Why is SVD more general than diagonalization?
2. What does the largest singular value measure?
3. Why is SVD useful for compression?
4. What is the relationship between $A^+$ and least squares?
5. What do right singular vectors mean in PCA?